In [1]:
## Restructured model with germany data

In [2]:
import pickle
with open('../datasets/data/polymod_germany.pkl', 'rb') as file:
    data = pickle.load(file)['contacts']

In [3]:
data

,id,age_part,sex_part,occ_part,edu_part,hh_size,class_size,work_contacts_nr,age_cnt,sex_cnt,y
0,846,10.0,F,5.0,2.0,4,26.0,NaN,43.0,M,3.0
1,846,10.0,F,5.0,2.0,4,26.0,NaN,70.0,F,2.0
2,846,10.0,F,5.0,2.0,4,26.0,NaN,68.0,M,2.0
3,846,10.0,F,5.0,2.0,4,26.0,NaN,11.0,F,1.0
4,846,10.0,F,5.0,2.0,4,26.0,NaN,13.0,F,2.0
...,...,...,...,...,...,...,...,...,...,...,...
10648,2185,70.0,M,2.0,8.0,2,NaN,NaN,57.0,M,1.0
10649,2185,70.0,M,2.0,8.0,2,NaN,NaN,57.0,M,1.0
10650,2185,70.0,M,2.0,8.0,2,NaN,NaN,61.0,M,1.0
10651,2185,70.0,M,2.0,8.0,2,NaN,NaN,32.0,M,1.0


In [4]:
from cntmosaic.dataloader.restru_loaders import MergedLoader

In [5]:
import pandas as pd
import numpy as np

In [6]:
import xarray as xr

In [7]:
from cntmosaic.dataloader import CoordToColumns

In [8]:
mapp = CoordToColumns(age_part='age_part', id_var='id', age_cnt='age_cnt', y='y')

In [9]:
d = MergedLoader(data, mapp)

In [10]:
d.load()

In [95]:
d.ds.id.groupby('age_part').nunique()


KeyError: 'age_part'

In [96]:
d.ds

<xarray.Dataset> Size: 74MB
Dimensions:   (id: 1281, age_part: 85, age_cnt: 85)
Coordinates:
  * id        (id) int64 10kB 846 847 848 849 850 ... 2180 2181 2182 2184 2185
  * age_part  (age_part) int64 680B 0 1 2 3 4 5 6 7 ... 77 78 79 80 81 82 83 84
  * age_cnt   (age_cnt) int64 680B 0 1 2 3 4 5 6 7 8 ... 77 78 79 80 81 82 83 84
Data variables:
    y         (id, age_part, age_cnt) int64 74MB 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0

In [65]:
d.ds.sum(dim=['id', 'age_cnt']).y.values


array([105, 170, 211, 210, 232, 263, 139, 149, 167,  86, 260, 159, 190,
       211, 273, 226, 321, 198, 355, 314, 194, 242, 302, 200, 134, 203,
       208, 153, 100,  88,  61,  83, 103, 149, 113, 110, 121, 117, 112,
       103, 177, 149, 174,  98, 188, 284, 160, 144, 171, 137, 150, 101,
       164, 193,  85, 125, 112,  52, 137, 117,  55,  99, 138,  62, 120,
       132,  72,  58, 100,  73,  50,  38,  42,  24,  17,  42,  11,  12,
         3,   0,   4,   6,  16,   7,   0])

In [14]:
df = d.ds.sum(dim=['id']).to_dataframe().reset_index()

In [ ]:
df2 = d.ds.sum(dim=['age_cnt']).to_dataframe().reset_index()
df2[df2.y!=0].groupby('age_part')['id'].nunique().reset_index(name='N')

In [22]:
df2

,id,age_part,y
0,846,0,0
1,846,1,0
2,846,2,0
3,846,3,0
4,846,4,0
...,...,...,...
108880,2185,80,0
108881,2185,81,0
108882,2185,82,0
108883,2185,83,0


In [17]:
df[df.y!=0].groupby('age_part').count()

,age_cnt,y
age_part,,
0,47,47
1,57,57
2,57,57
3,55,55
4,65,65
...,...,...
78,3,3
80,3,3
81,5,5


In [24]:
from cntmosaic.preprocess import make_train_data
old = make_train_data(data, 'id')

In [25]:
old

,age_part,age_cnt,y,N
0,0,0,0,13
1,0,1,2,13
2,0,2,6,13
3,0,3,0,13
4,0,4,0,13
...,...,...,...,...
7050,83,80,0,2
7051,83,81,0,2
7052,83,82,1,2
7053,83,83,0,2


In [ ]:
d.raw_df.y.sum()

In [ ]:
d.raw_df

In [ ]:
d.ds.sel(id=846, age_part=10, age_cnt=43)

In [ ]:
d.ds.groupby(['age_part', 'age_cnt'])

In [ ]:
from cntmosaic.models.restr_BRCfine import restr_BRCfine

In [ ]:
model = restr_BRCfine(out)

In [ ]:
model.compile()

In [ ]:
import jax
model.run_inference_mcmc(jax.random.PRNGKey(0),
    num_samples = 50,
    num_warmup = 50,
    num_chains = 2)

In [ ]:
from cntmosaic.sim import ModelEvaluatorMCMC

In [ ]:
e = ModelEvaluatorMCMC(model)

In [ ]:
e.post = model.mcmc.get_samples(group_by_chain=True)
#e.get_posterior() extra dimension for different chains

In [ ]:
import numpy as np

In [ ]:
log_rate = e.post['log_rate']
post_cint = {}
for name, site in e.post.items():
    if 'log_delta' in name:
        var = name.split('/')[0]
        cat = e.model.data[var].cat.categories
        post_cint[var] = {
            cat[i]: np.exp(log_rate[:, None, :, :] + site + e.model._precompute.log_P[None, None, :, :])[:, i, :, :]
            for i in range(len(cat))
        }

In [ ]:
e.post.keys()

In [ ]:
e.post['inv_disp'].shape

In [ ]:
import numpyro
import jax.numpy as jnp
from numpyro import distributions as dist

def guide():
    # --- beta0: scalar Normal(0, 10) ---
    beta0_loc = numpyro.param("beta0_loc", jnp.zeros(()))
    beta0_scale = numpyro.param("beta0_scale", jnp.ones(()), constraint=dist.constraints.positive)
    numpyro.sample("beta0", dist.Normal(beta0_loc, beta0_scale))

    # --- alpha: scalar InverseGamma(5, 5) ---
    alpha_conc = numpyro.param("alpha_conc", jnp.ones(()), constraint=dist.constraints.positive)
    alpha_rate = numpyro.param("alpha_rate", jnp.ones(()), constraint=dist.constraints.positive)
    numpyro.sample("baseline", dist.InverseGamma(alpha_conc, alpha_rate))

    # --- rho: vector of 2 InverseGamma(5, 5) ---
    rho_conc = numpyro.param("rho_conc", jnp.ones(2), constraint=dist.constraints.positive)
    rho_rate = numpyro.param("rho_rate", jnp.ones(2), constraint=dist.constraints.positive)
    numpyro.sample("rho", dist.InverseGamma(rho_conc, rho_rate))

    inv_disp_rate = numpyro.param("inv_disp_rate", jnp.ones(len(out)), constraint=dist.constraints.positive)
    numpyro.sample("inv_disp", dist.Exponential(inv_disp_rate))


In [ ]:
model.run_inference_svi(jax.random.PRNGKey(0),guide, num_steps = 1000)

In [ ]:
from cntmosaic.sim import ModelEvaluatorSVI
svi = ModelEvaluatorSVI(model)

In [ ]:
svi.get_pred_cint()

In [ ]:
svi.pred_cint